In [ ]:
import operator
from typing import Annotated, List, TypedDict
from langgraph.graph import StateGraph, END

# 1. 상태 정의 (노드 간 공유 정보)
class ClusterState(TypedDict):
    task: str
    specs: str
    code: str
    test_results: List[str]
    is_code_valid: bool
    security_check: bool
    final_approval: bool

# 2. 노드 함수 정의 (자율적 상호작용)
def router_node(state: ClusterState):
    """시작점에서 기획과 보안 검사를 동시에 가동 준비"""
    print("[Router] 프로젝트 시작: 기획 및 보안 가이드라인 수립")
    return {"task": "Full Stack App"}

def planner_node(state: ClusterState):
    print("[Planner] 기술 명세서 작성 중...")
    return {"specs": "Auth API + Database Schema"}

def security_node(state: ClusterState):
    print("[Security] 보안 취약점 사전 스캔 중...")
    return {"security_check": True}

def developer_node_1(state: ClusterState):
    """Planner의 specs를 받아 코드 작성 1"""
    print(f"[Developer] '{state['specs']}' 기반 코딩 시작")
    return {"code": "def login(): pass"}

def developer_node_2(state: ClusterState):
    """Planner의 specs를 받아 코드 작성 2"""
    print(f"[Developer] '{state['specs']}' 기반 코딩 시작")
    return {"code": "def login(): pass"}

def tester_node(state: ClusterState):
    """Developer의 code를 검증하여 직접 피드백"""
    print("[Tester] 유닛 테스트 실행 중...")
    valid = len(state['code']) > 0
    return {"is_code_valid": valid, "test_results": ["Success" if valid else "Fail"]}

def reviewer_node(state: ClusterState):
    """최종 승인 노드"""
    print("[Reviewer] 코드 및 보안 문서 최종 검토")
    return {"final_approval": state['is_code_valid'] and state['security_check']}

def deploy_node(state: ClusterState):
    print("[Deployer] 상용 환경 배포 완료!")
    return {}

# 3. 그래프 구성
builder = StateGraph(ClusterState)

builder.add_node("router", router_node)
builder.add_node("planner", planner_node)
builder.add_node("security", security_node)
builder.add_node("developer_1", developer_node_1)
builder.add_node("developer_2", developer_node_2)
builder.add_node("tester", tester_node)
builder.add_node("reviewer", reviewer_node)
builder.add_node("deploy", deploy_node)

# 4. 엣지 연결 (상호작용 중심)
builder.set_entry_point("router")

# 병렬 실행 (Parallel): Router -> Planner & Security 동시 시작
builder.add_edge("router", "planner")
builder.add_edge("router", "security")

# 순차 흐름: Planner -> Developer
builder.add_edge("planner", "developer_1")
builder.add_edge("planner", "developer_2")

# 피드백 루프 (Direct Interaction): Developer <-> Tester
builder.add_edge("developer_1", "tester")
builder.add_edge("developer_2", "tester")
builder.add_conditional_edges(
    "tester",
    lambda x: "developer_1" if not x["is_code_valid"] else "reviewer",
    {"developer_1": "developer_1", "reviewer": "reviewer"}
)

# 보안과 테스트가 모두 Reviewer로 집결 (Fan-in)
builder.add_edge("security", "reviewer")
builder.add_edge("reviewer", "router")

# 최종 리뷰 Reviewer -> Deploy or End
builder.add_conditional_edges(
    "reviewer",
    lambda x: "deploy" if x["final_approval"] else END,
    {"deploy": "deploy", END: END}
)
builder.add_edge("deploy", END)

app = builder.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. 상태 정의 (노드 간 공유 정보)
class ClusterState(TypedDict):
    query: str
    documents: list[str]
    response: str
    response_grade: str # "back to web searcher", "back to query", "pass"

# 2. 노드 함수 정의 (자율적 상호작용)
def query_refiner_node(state: ClusterState):
    """사용자의 질문을 정제하는 노드"""
    return {"query": "refined Query"}

def retriever_node(state: ClusterState):
    """저장된 문서에서 검색을 수행하는 노드"""
    return {"documents": ["doc1", "doc2"]}

def llm_inference_node(state: ClusterState):
    """LLM 답변을 생성하는 노드"""
    return {"response": "LLM 답변"}

def response_refiner_node(state: ClusterState):
    """최종 답변을 정제하는 노드"""
    return {"response": "Refined LLM Response"}

def query_rewriter_node(state: ClusterState):
    """사용자의 질문을 다시 쓰는 노드"""
    return {"query": "rewrited query"}

def web_searcher_node(state: ClusterState):
    """자료가 부족할 경우 웹에서 검색을 수행하는 노드"""
    return {"documents": ["doc1", "doc2", "web_doc"]}

def response_evaluator_node(state: ClusterState):
    """답변 평가 노드"""
    return {}

def response_node(state: ClusterState):
    """답변 노드"""
    return {}

# 3. 그래프 구성
builder = StateGraph(ClusterState)

builder.add_node("query_refiner", query_refiner_node)
builder.add_node("retriever", retriever_node)
builder.add_node("llm", llm_inference_node)
builder.add_node("response_refiner", response_refiner_node)
builder.add_node("query_rewriter", query_rewriter_node)
builder.add_node("web_searcher", web_searcher_node)
builder.add_node("response_evaluator", response_evaluator_node)
builder.add_node("response", response_node)

# 4. 엣지 연결 (상호작용 중심)
builder.set_entry_point("query_refiner")

# 병렬 실행 (Parallel)
builder.add_edge("query_refiner", "retriever")
builder.add_edge("query_refiner", "web_searcher")

# 병렬 실행 집결 (Fan-in)
builder.add_edge("retriever", "llm")
builder.add_edge("web_searcher", "llm")

# 피드백 루프
builder.add_edge("llm", "response_evaluator")

# 분기 노드 (평가)
builder.add_conditional_edges(
    "response_evaluator",
    lambda x: "web_searcher" if x["response_grade"] == "back to web searcher" else\
              "query_rewriter" if x["response_grade"] == "back to query" else\
              "response_refiner" if x["response_grade"] == "pass" else "",
    {"web_searcher": "web_searcher", "query_rewriter": "query_rewriter", "response_refiner":"response_refiner"}
)

# 피드백 이후 엣지 연결
builder.add_edge("query_rewriter", "query_refiner")
builder.add_edge("response_refiner", "response")

app = builder.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))